In [13]:
import os
from docx import Document

ss1_folder = r'C:\Users\charlott\Dropbox (Personal)\Paper_Corporate_Obstructionism\Data\LexisNexis\ss1'

merged_doc = Document()

def append_docx_to_merged(file_path, merged_doc):
    if os.path.exists(file_path):
        try:
            doc = Document(file_path)
            for para in doc.paragraphs:
                merged_doc.add_paragraph(para.text)
            merged_doc.add_page_break()  
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

for file in os.listdir(ss1_folder):
    if file.endswith('.DOCX') or file.endswith('.docx'):
        file_path = os.path.join(ss1_folder, file)
        append_docx_to_merged(file_path, merged_doc)

for subdir in os.listdir(ss1_folder):
    subdir_path = os.path.join(ss1_folder, subdir)
    
    if os.path.isdir(subdir_path):
        for file in os.listdir(subdir_path):
            if file.endswith('.DOCX') or file.endswith('.docx'):
                file_path = os.path.join(subdir_path, file)
                append_docx_to_merged(file_path, merged_doc)

base_folder = r'C:\Users\charlott\Dropbox (Personal)\Paper_Corporate_Obstructionism\Data\LexisNexis'

output_file = os.path.join(base_folder, "ss1_merged.docx")
merged_doc.save(output_file)

Error reading C:\Users\charlott\Dropbox (Personal)\Paper_Corporate_Obstructionism\Data\LexisNexis\ss1\~$g Oil Climate Ruling Sets Dangerous Liability Precedent(2).DOCX: Package not found at 'C:\Users\charlott\Dropbox (Personal)\Paper_Corporate_Obstructionism\Data\LexisNexis\ss1\~$g Oil Climate Ruling Sets Dangerous Liability Precedent(2).DOCX'
Error reading C:\Users\charlott\Dropbox (Personal)\Paper_Corporate_Obstructionism\Data\LexisNexis\ss1\~$ionism, Militarism, and the Decline of US Power_ - Book Review by Stephen Lendman.DOCX: Package not found at 'C:\Users\charlott\Dropbox (Personal)\Paper_Corporate_Obstructionism\Data\LexisNexis\ss1\~$ionism, Militarism, and the Decline of US Power_ - Book Review by Stephen Lendman.DOCX'


Go into R and convert Word documents into Excel documents using the specific (LN_Uni_Cleaning_1)

In [17]:
import pandas as pd   

file_path = 'C:/Users/charlott/Dropbox (Personal)/Paper_Corporate_Obstructionism/Data/LexisNexis/ss1_merged.xlsx'  

meta_df = pd.read_excel(file_path, sheet_name='Meta Data')
articles_df = pd.read_excel(file_path, sheet_name='Articles Data')
paragraphs_df = pd.read_excel(file_path, sheet_name='Paragraphs Data')

print(meta_df.head())
print(articles_df.head())

   ID                                        Source_File  \
0   1  C:/Users/charlott/Dropbox (Personal)/Paper_Cor...   
1   2  C:/Users/charlott/Dropbox (Personal)/Paper_Cor...   
2   3  C:/Users/charlott/Dropbox (Personal)/Paper_Cor...   
3   4  C:/Users/charlott/Dropbox (Personal)/Paper_Cor...   
4   5  C:/Users/charlott/Dropbox (Personal)/Paper_Cor...   

                 Newspaper       Date      Length Section  \
0           Elder of Ziyon 2023-08-28  8422 words     NaN   
1           Elder of Ziyon 2023-09-08  9006 words     NaN   
2           Elder of Ziyon 2021-10-14  7779 words     NaN   
3             Coyote Gulch 2024-05-11  3065 words     NaN   
4  July 10, 2024 Wednesday 2024-07-10   876 words     NaN   

                Author  Edition             Headline  Graphic  
0                  Ian      NaN        Newstex Blogs    False  
1                  Ian      NaN        Newstex Blogs    False  
2                  Ian      NaN        Newstex Blogs    False  
3         Coyote

In [18]:
print(paragraphs_df.head())

   Art_ID  Par_ID                                          Paragraph
0       1       1               August 28th, 2023 ( — Delivered by )
1       1       2                                          From Ian:
2       1       3   The very good news is that, well-financed and...
3       1       4  There are two trajectories that define the upc...
4       1       5  The other trajectory that the majority of the ...


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(articles_df['Article'])

cosine_sim = cosine_similarity(tfidf_matrix)

threshold = 0.90
duplicates = set()

for i in range(len(cosine_sim)):
    for j in range(i + 1, len(cosine_sim)):
        if cosine_sim[i][j] >= threshold:
            duplicates.add(j)

mask = ~articles_df.index.isin(duplicates)

filtered_articles_df = articles_df[mask]
print(f"Number of articles before removal: {len(articles_df)}")
print(f"Number of articles after removal: {len(filtered_articles_df)}")

print(filtered_articles_df.head())

Number of articles before removal: 89
Number of articles after removal: 68
   ID                                            Article
0   1  August 28th, 2023 ( — Delivered by )\nFrom Ian...
1   2  September 8th, 2023 ( — Delivered by )\nFrom I...
2   3  Oct 14, 2021( Elder of Ziyon: Delivered by New...
3   4  May 11, 2024( Coyote Gulch: https://coyotegulc...
4   5  The following information was released by the ...


#### Filter out years 

(before doublespeak documents have been published)

In [33]:
initial_row_count = len(data)

merged_df = pd.merge(filtered_articles_df, meta_df, on='ID', how='inner')
data['Date'] = data['Date'].astype(str)
data = data[data['Date'].str.startswith('2024')]
final_row_count = len(data)
dropped_rows = initial_row_count - final_row_count

print(f"Number of rows initially: {initial_row_count}")
print(f"Number of rows after filtering: {final_row_count}")
print(f"Number of rows dropped: {dropped_rows}")

Number of rows initially: 53
Number of rows after filtering: 53
Number of rows dropped: 0


In [28]:
print(data['Date'].str.strip().unique())

['2024-05-11' '2024-07-10' '2024-09-12' '2024-09-11' '2024-09-26'
 '2024-08-13' '2024-05-01' '2024-10-11' '2024-06-17' '2024-05-23'
 '2024-09-05' '2024-05-06' '2024-04-29' '2024-05-22' '2024-05-02'
 '2024-04-30' '2024-05-30' '2024-05-15' '2024-05-04' '2024-05-09'
 '2024-05-03' '2024-06-11' '2024-06-02' '2024-04-25' '2024-05-31'
 '2024-05-29' '2024-06-05']


In [36]:
ssn1_2024 = data[['ID', 'Article']]

file_path = r'C:/Users/charlott/Dropbox (Personal)/Paper_Corporate_Obstructionism/Data/LexisNexis/ss1_merged_2024.xlsx'
ssn1_2024.to_excel(file_path, index=False)

Open open excel file and do a manual inspection by reading the articles, then reduce further. Decided to limit to articles that mention 'committee'. Example ID 4: "Earlier this year, Democrats on the House Oversight Committee and Senate Budget Committee released a joint report accusing the fossil fuel industry of "climate denial, disinformation, and doublespeak." The report noted that oil and gas companies have relied on industry groups, including the American Petroleum Institute, to spread misleading information and to lobby for unpopular proposals that they did not want to be associated with."

In [45]:
mention_committe = data[data['Article'].str.contains('committee', case=False, na=False)]
print(mention_committe['ID'].unique())

data = mention_committe

[ 4  5  7  9 10 12 13 14 16 17 20 21 23 24 26 32 35 36 37 38 40 43 47 49
 50 51 53 54 57 58 59 60 61 63 70 71 73 74 75 76 78 88]


#### Run NER  using spacy

Categorize only specific types of entities (GPE, ORG, FAC, PERSON, LOC, WORK_OF_ART, and LAW), other, such as date and time, are irrelevant

In [46]:
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

def perform_ner(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

data['Named_Entities'] = data['Article'].apply(perform_ner)

print(data[['Article', 'Named_Entities']].head())

                                             Article  \
3  May 11, 2024( Coyote Gulch: https://coyotegulc...   
4  The following information was released by the ...   
6  Across the country, judges have largely agreed...   
8  August 13th, 2024 ( — Delivered by )\nA new re...   
9  Public pledges from the world's largest oil co...   

                                      Named_Entities  
3  [(May 11, 2024, DATE), (United States, GPE), (...  
4  [(the Center for Responsive Politics, ORG), (J...  
6  [(Circuit Court, ORG), (Baltimore City, GPE), ...  
8  [(August 13th, 2024, DATE), (Environmental Def...  
9  [(Democratic, NORP), (the Senate Budget Commit...  


In [47]:
from collections import defaultdict

target_categories = {"GPE", "ORG", "FAC", "PERSON", "LOC", "WORK_OF_ART", "LAW"}

for entities in data['Named_Entities']:
    for entity in entities:
        entity_text, entity_label = entity  
        if entity_label in target_categories:
            entities_by_category[entity_label].add(entity_text) 

for category, entities in entities_by_category.items():
    print(f"\nCategory: {category} (Total: {len(entities)})")
    print(sorted(entities)) 


Category: DATE (Total: 521)
["' first month", '+WASE035+', '1-21', '1.45C', '10 years', '10 years earlier', '10.1007', '100 years', '100 years ago', '100-year', '1067-2024', '1099', '11', '1122-1128', '1189-1202', '12 June 2019', '12 to 20,000 years ago', '1325-1337', '14, 4431', '15 years', '15, 119401', '150 years ago', '16-year-old', '18 January 2018', '19', '1930', '1940', '1950s', '1959', '1959 - 65 years ago', '1961-2100', '1965', '1967', '1968', '1970', '1970s', '1971-2021', '1975', '1977', '1977-2014', '1978', '1979', '1980', '1981', '1982', '1985', '1986', '1986 - 2015', '1986 to 2015', '1988', '1988-89', '1989', '1991', '1993', '1995', '1996', '1997', '1998', '1998-2020', '1999', '20 years', '20 years from now', '20,000 years', '20-year', '2000', '2001', '2003', '2004', '2004 and 2005', '2004-06', '2005', '2006', '2006-08', '2007', '2008', '2008 to 2018', '2008-2022', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2017-20', '2018', '2019', '2019 to'

In [48]:
category_dfs = {}
for category, entities in entities_by_category.items():
    category_dfs[category] = pd.DataFrame(sorted(entities), columns=["Entity"])
output_file = "NER_on_Lexisnexis_search_query_results.xlsx"
with pd.ExcelWriter(output_file) as writer:
    for category, df in category_dfs.items():
        df.to_excel(writer, sheet_name=category, index=False)

Excel file saved as 'NER_on_Lexisnexis_search_query_results.xlsx' with each category as a separate sheet.


Done, now filter the entities manually.

### Read in file with manually filtered entities
It has 20 duplicates, remove them.

In [8]:
import pandas as pd

file_path = "NER_on_Lexisnexis_search_query_results_filtered_CG_CB_TK.xlsx"
sheets = pd.read_excel(file_path, sheet_name=None)

parts = []

for sheet_name, df in sheets.items():
    # from Entities_filtered
    if "Entities_filtered" in df.columns:
        a = (
            df[["Entities_filtered"]]
            .dropna(subset=["Entities_filtered"])
            .assign(Sub_category=pd.NA)
        )
        a["Entities_filtered"] = a["Entities_filtered"].astype(str).str.strip()
        parts.append(a)

    # from Additional_keywords -> flag as Keyword Climate Environment
    if "Additional_keywords" in df.columns:
        b = (
            df[["Additional_keywords"]]
            .dropna(subset=["Additional_keywords"])
            .rename(columns={"Additional_keywords": "Entities_filtered"})
            .assign(Sub_category="Keyword Climate Environment")
        )
        b["Entities_filtered"] = b["Entities_filtered"].astype(str).str.strip()
        parts.append(b)

# append (no sheets structure)
entities_df = pd.concat(parts, ignore_index=True)

# optional: drop empty strings
entities_df = entities_df[entities_df["Entities_filtered"].ne("")]

# optional: de-duplicate by entity, keeping the keyword label if it exists
entities_df = (
    entities_df.sort_values("Sub_category", na_position="first")
               .drop_duplicates(subset=["Entities_filtered"], keep="last")
               .reset_index(drop=True)
)

# preprocessing: remove trailing blanks
str_cols = entities_df.select_dtypes(include="object").columns
entities_df[str_cols] = entities_df[str_cols].apply(lambda c: c.str.strip())


print("Count, after removing duplicates:", len(entities_df))
print(
    entities_df["Sub_category"]
    .eq("Keyword Climate Environment")
    .sum()
)
print("LexisNexis_Entities columns:", entities_df.columns.tolist())

Count, after removing duplicates: 700
58
LexisNexis_Entities columns: ['Entities_filtered', 'Sub_category']


In [9]:
entities_df.to_excel("LexisNexis_Entities.xlsx", index=False)
print("Saved: LexisNexis_Entities.xlsx")

Saved: LexisNexis_Entities.xlsx


In [10]:
# Now append this in the Creation_NER_Codebook python script.